# Exercise 66 — `pandas.read_sql` and `DataFrame.to_sql`

## Concept

For anything tabular, pandas integrates directly with DB-API connections — no manual cursor handling.

### Read

```python
import pandas as pd
import sqlite3

conn = sqlite3.connect("app.db")

df = pd.read_sql("SELECT * FROM orders", conn)
df = pd.read_sql("SELECT * FROM orders WHERE amount > ?", conn, params=(100,))
```

`pd.read_sql` accepts:
- A SQL string
- A connection (sqlite3, SQLAlchemy, psycopg2, etc.)
- Optional `params=` for parameterized queries (still no f-strings!)
- `chunksize=` to stream large results back as an iterator of DataFrames

### Write

```python
df.to_sql(
    "orders",
    conn,
    if_exists="append",     # 'fail' (default), 'replace', or 'append'
    index=False,            # almost always what you want — don't write the DataFrame index as a column
    chunksize=1000,         # batch size for inserts
)
```

`if_exists`:
- `"fail"` — raise if table exists (default — paranoid, but safe)
- `"replace"` — drop and recreate the table (destructive, use carefully)
- `"append"` — insert into existing table

### When to use this vs raw `sqlite3`

- **DataFrame → DB**: use `to_sql` (handles the schema + executemany for you).
- **DB → DataFrame**: use `read_sql` — simpler than cursor + `DataFrame(rows)`.
- **Single-row CRUD / fine-grained transactions**: stick with raw `sqlite3` — pandas is overkill.
- **Heavy ETL with row-by-row logic**: raw cursors give you more control.

### Streaming large reads

```python
for chunk in pd.read_sql("SELECT * FROM huge", conn, chunksize=10_000):
    process(chunk)            # one DataFrame at a time, memory stays flat
```

## Setup

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB = "pandas_sql.db"
Path(DB).unlink(missing_ok=True)

seed = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7],
    "customer": ["Alice", "Bob", "Alice", "Carol", "Bob", "Alice", "Carol"],
    "amount":   [120.50, 45.00, 300.75, 89.99, 210.00, 55.25, 430.10],
    "paid":     [True, True, False, True, True, False, True],
})

with sqlite3.connect(DB) as conn:
    seed.to_sql("orders", conn, if_exists="replace", index=False)
print("seeded", DB)


## Task 1 — `read_sql` to DataFrame

Write a function that returns the full `orders` table as a DataFrame.

```python
def read_all_orders(db_path: str) -> pd.DataFrame:
    ...
```

Open and close the connection inside the function. Verify shape and a couple of cells.

In [ ]:
import sqlite3
import pandas as pd

def read_all_orders(db_path: str) -> pd.DataFrame:
    pass  # TODO

df = read_all_orders(DB)
assert df.shape == (7, 4)
assert df.loc[0, "customer"] == "Alice"
assert df["amount"].sum() == seed["amount"].sum()
print("ok")


## Task 2 — Parameterized `read_sql`

Write a function that returns orders **above a given threshold**, using `params=` (NOT f-strings!).

```python
def read_orders_above(db_path: str, threshold: float) -> pd.DataFrame:
    ...
```

Expected: `read_orders_above(DB, 200)` returns 3 rows (order_ids 3, 5, 7).

In [ ]:
import sqlite3
import pandas as pd

def read_orders_above(db_path: str, threshold: float) -> pd.DataFrame:
    pass  # TODO

result = read_orders_above(DB, 200)
assert set(result["order_id"]) == {3, 5, 7}
print("ok")


## Task 3 — `to_sql` append

Write a function that takes a DataFrame of new orders and **appends** them to the existing `orders` table.

```python
def append_orders(db_path: str, new_orders: pd.DataFrame) -> int:
    """Append new_orders to the 'orders' table. Return the total row count after appending."""
```

Use `if_exists="append"` and `index=False`. Verify the new row count.

In [ ]:
import sqlite3
import pandas as pd

new = pd.DataFrame({
    "order_id": [8, 9],
    "customer": ["Diana", "Eve"],
    "amount":   [99.0, 12.0],
    "paid":     [True, False],
})

def append_orders(db_path: str, new_orders: pd.DataFrame) -> int:
    pass  # TODO

total = append_orders(DB, new)
assert total == 9

with sqlite3.connect(DB) as conn:
    after = pd.read_sql("SELECT * FROM orders ORDER BY order_id", conn)
assert after["order_id"].tolist() == [1, 2, 3, 4, 5, 6, 7, 8, 9]
assert after.iloc[-1]["customer"] == "Eve"
print("ok")


## Task 4 — Streaming large reads with `chunksize`

Use `chunksize=` to iterate the result in chunks. Compute the total `amount` without ever holding all rows in memory at once.

```python
def sum_amount_chunked(db_path: str, chunksize: int) -> float:
    ...
```

Return the sum. For this small dataset the behavior is identical to a single read; the test is just verifying you used the iterator API correctly.

Hint: `pd.read_sql(..., chunksize=N)` returns an **iterator** of DataFrames, not a single DataFrame. Iterate and sum.

In [ ]:
import sqlite3
import pandas as pd

def sum_amount_chunked(db_path: str, chunksize: int) -> float:
    pass  # TODO

total = sum_amount_chunked(DB, chunksize=3)
with sqlite3.connect(DB) as conn:
    expected = pd.read_sql("SELECT SUM(amount) AS s FROM orders", conn)["s"].iloc[0]
assert total == expected
print("ok")
